# 41. FP8 and KV Cache Quantization | FP8 与 KV Cache 量化
**难度：** Hard | **环境：** CPU-first | **标签：** `量化压缩`, `FP8`, `KV Cache` | **目标人群：** 量化压缩学习者

---

## 本节导读

第 40 节关注的是权重量化：把模型参数压得更小，降低加载和访存成本。但推理阶段的压力不只来自权重。长上下文生成时，KV Cache 会随着序列长度和并发请求持续增长；同时，部分激活或中间张量也会带来带宽压力。只压权重，不能完全解决长上下文推理的显存和带宽瓶颈。

本节分别讨论低精度运行时张量与分组 KV Cache：它们都需要保存量化值和 scale，但面对的动态范围、生命周期和读写方式不同。学完后，你应该能看清“量化值、scale、反量化、误差检查”这条闭环，并理解为什么 KV Cache 通常按最后一维分组处理。

这让量化对象从模型权重进一步扩展到推理过程中持续生成和读取的状态。

**关键词：** `FP8`, `KV cache quantization`, `deployment`


---


## 前置阅读

**导语：** 先区分权重、运行时张量和 KV Cache 这三类对象，再观察 FP8 与 KV Cache 量化如何改变存储、带宽和误差。
- [核心前置：22 vLLM PagedAttention | vLLM 分页注意力](./22_vLLM_PagedAttention.ipynb)
- [核心前置：25 Quantization W8A16 | W8A16 量化](./25_Quantization_W8A16.ipynb)
- [可选扩展：40 GPTQ and AWQ Weight Quantization | GPTQ 与 AWQ 权重量化](./40_GPTQ_and_AWQ_Weight_Quantization.ipynb)

---


### Step 1: 运行时量化如何改变推理状态

生成式推理会持续追加 KV Cache；上下文越长、并发越高，缓存越容易成为显存容量和带宽压力。本节从运行时量化出发，追踪“浮点状态 → 量化值与 scale → 恢复结果”的闭环。量化后既要关注缓存字节数，也要观察恢复误差和额外处理时间。

先区分普通激活与 KV Cache：前者通常随一次算子执行短暂存在，后者会跨多个 Decode 步持续读取和追加，因此需要不同的量化粒度与状态记录。

| 量化对象 | 发生时机 | 主要成本 | 需要保留的状态 |
|---|---|---|---|
| 普通激活 / 中间张量 | 算子执行或数据传递过程中 | 存储与带宽 | 低精度值、scale、原始 shape |
| KV Cache | 请求执行过程中持续追加和读取 | 长上下文显存与缓存带宽 | 分组量化值、每组 scale、缓存 shape |
| 共同验证 | 量化后恢复时 | 误差与额外处理时间 | 恢复结果、MSE、耗时和字节数 |

![FP8 与 KV Cache 量化路径](../docs/public/02_PyTorch_Algorithms/41_fp8_kv_quant_flow_cn.svg)


### Step 2: FP8 近似量化如何保存和恢复张量

本节用 INT8 容器模拟低精度张量的保存路径：先根据 `absmax` 得到 scale，再把浮点值映射为整数，恢复时用同一个 scale 还原近似值。可以把它写成 `q = round(x * scale)`、`x_hat = q / scale`；因此 scale、量化值和原始 shape 缺一不可。

这里重点观察动态范围变化如何影响量化误差和存储字节数。真实 FP8 不只是把 FP32 截成更少的 bit，还要选择格式、保存 scale，并确认输入、累加和输出 dtype 的组合；不同硬件和 backend 可能选择不同的执行路径。

| 维度 | E4M3 | E5M2 | 对推理路径的影响 |
|---|---|---|---|
| 指数位 / 尾数位 | 4 / 3 | 5 / 2 | 精度与动态范围取舍不同 |
| 更关注的场景 | 前向计算、权重或激活表示 | 梯度或动态范围更大的数值 | 不能只看存储 bit 数 |
| 需要记录 | dtype、scale、硬件支持 | dtype、scale、硬件支持 | 判断是否进入目标 kernel |


### Step 3: KV Cache 如何按分组量化并读取

KV Cache 沿最后一维分组，每组保存一套 scale。`kv_group_size` 越小，scale 对局部数值范围的适应性通常越强，但元数据和量化处理次数也会增加。追加新 token 时按同一规则写入量化值与 scale；读取历史状态时也必须按同一分组定位二者。

量化改变每个状态的表示；分页、前缀复用和驱逐改变状态如何分配、复用或保留。它们可以组合，但回答的是不同问题。

| 因素 | 改变的状态 | 主要观察 |
|---|---|---|
| `seq_len`、layers、KV heads、head dim | 缓存总容量 | 长上下文压力与原始字节数 |
| `kv_group_size` | scale 数量与局部误差 | 压缩比、MSE、量化/恢复耗时 |
| dtype / 容器位宽 | 每个量化值的存储 | 原始字节、量化字节与 scale 元数据 |
| 分页、Prefix Cache、驱逐 | 分配、复用与驻留状态 | 与量化互补，不直接改变单元素位宽 |

![KV Cache 分组量化机制](../docs/public/02_PyTorch_Algorithms/41_kv_cache_quantization_mechanism_cn.svg)


### Step 4：实现并验证运行时量化状态

题目区围绕三个机制完成运行时量化闭环：先把浮点状态映射为低精度值与 scale，再沿 KV Cache 最后一维建立分组状态，最后按同一分组规则恢复。shape 保存和 MSE 计算由骨架负责，测试会检查这些状态是否一致。

| TODO | 实现对象 | 机制责任 | 关键测试 |
|:---|:---|:---|:---|
| TODO 1 | `_sym_quantize` | 由动态范围生成安全 scale，并得到有界低精度值 | 全零输入、INT8 dtype、有限恢复值 |
| TODO 2 | `quantize_kv_cache` | 沿最后一维向上取整分组，为每组保存 scale | 非整除边界、scale 形状、原始 shape |
| TODO 3 | `dequantize_kv_cache` | 使用当前位置的 group scale 恢复 KV 区间 | 恢复形状、有限值、低误差 |
| 骨架状态 | `fp8_shape`、`kv_shape`、MSE | 追踪量化对象并统一评估恢复误差 | 状态契约与重复比较 |


In [ ]:
import torch
import torch.nn as nn


In [ ]:
# 题目区只挖空三个机制：对称量化、KV Cache 分组和按组恢复。
# shape 状态与 MSE 由骨架保存，测试用它们检查量化闭环。

class FP8KVCacheSim(nn.Module):
    """保存量化值、scale 和 shape 的最小运行时量化模拟器。

    `fp8_qmax` 是量化整数范围上限，`kv_group_size` 是 KV Cache 最后一维共享
    一套 scale 的元素数，`eps` 用于避免全零或极小动态范围下的除零。
    """

    def __init__(self, fp8_qmax: int = 127, kv_group_size: int = 64, eps: float = 1e-8):
        super().__init__()
        if kv_group_size <= 0:
            raise ValueError("kv_group_size must be positive")
        self.fp8_qmax = fp8_qmax
        self.kv_group_size = kv_group_size
        self.eps = eps

        self.register_buffer("fp8_q", torch.empty(0, dtype=torch.int8), persistent=False)
        self.register_buffer("fp8_scale", torch.tensor(1.0), persistent=False)
        self.register_buffer("kv_q", torch.empty(0, dtype=torch.int8), persistent=False)
        self.register_buffer("kv_scale", torch.empty(0), persistent=False)
        self.fp8_shape = None
        self.kv_shape = None

    def _sym_quantize(self, x: torch.Tensor, qmax: int):
        x = x.detach().float()
        # TODO 1（对称量化）：由动态范围生成安全 scale，并映射为 [-qmax, qmax] 的 INT8 值。
        # absmax = ???
        # scale = ???
        # q = ???
        return q, scale

    def _sym_dequantize(self, q: torch.Tensor, scale: torch.Tensor):
        return q.to(scale.dtype) / scale.clamp_min(self.eps)

    def quantize_fp8(self, x: torch.Tensor):
        q, scale = self._sym_quantize(x, self.fp8_qmax)
        self.fp8_q = q
        self.fp8_scale = scale
        self.fp8_shape = tuple(x.shape)
        return q, scale

    def dequantize_fp8(self):
        if self.fp8_shape is None:
            raise RuntimeError("Call quantize_fp8() before dequantize_fp8().")
        return self._sym_dequantize(self.fp8_q, self.fp8_scale)

    def quantize_kv_cache(self, kv_cache: torch.Tensor):
        kv = kv_cache.detach().float()
        if kv.ndim < 2:
            raise ValueError("KV cache should have at least 2 dimensions.")

        last_dim = kv.size(-1)
        # TODO 2（KV 分组）：对最后一维向上取整，最后一组可以不足 kv_group_size。
        # n_groups = ???
        qkv = torch.zeros_like(kv, dtype=torch.int8)
        scales = torch.zeros(kv.shape[:-1] + (n_groups,), dtype=kv.dtype, device=kv.device)

        flat = kv.reshape(-1, last_dim)
        flat_q = qkv.reshape(-1, last_dim)
        flat_scale = scales.reshape(-1, n_groups)

        for row in range(flat.size(0)):
            for g in range(n_groups):
                start = g * self.kv_group_size
                end = min(start + self.kv_group_size, last_dim)
                chunk = flat[row, start:end]
                if chunk.numel() == 0:
                    continue
                q, scale = self._sym_quantize(chunk, self.fp8_qmax)
                flat_q[row, start:end] = q
                flat_scale[row, g] = scale

        self.kv_q = qkv
        self.kv_scale = scales
        self.kv_shape = tuple(kv.shape)
        return qkv, scales

    def dequantize_kv_cache(self):
        if self.kv_shape is None:
            raise RuntimeError("Call quantize_kv_cache() before dequantize_kv_cache().")

        kv = self.kv_q.to(self.kv_scale.dtype)
        last_dim = kv.size(-1)
        n_groups = self.kv_scale.size(-1)
        flat = kv.reshape(-1, last_dim)
        flat_out = torch.zeros_like(flat, dtype=self.kv_scale.dtype)
        flat_scale = self.kv_scale.reshape(-1, n_groups)

        for row in range(flat.size(0)):
            for g in range(n_groups):
                start = g * self.kv_group_size
                end = min(start + self.kv_group_size, last_dim)
                scale = flat_scale[row, g]
                # TODO 3（按组恢复）：只用当前 group 的 scale 恢复对应区间。
                # restored_chunk = ???
                flat_out[row, start:end] = restored_chunk

        return flat_out.reshape(self.kv_shape)

    def fit(self, hidden_states: torch.Tensor, kv_cache: torch.Tensor | None = None):
        self.quantize_fp8(hidden_states)
        if kv_cache is not None:
            self.quantize_kv_cache(kv_cache)
        return self

    def forward(self, hidden_states: torch.Tensor, kv_cache: torch.Tensor | None = None):
        fp8_q, fp8_scale = self._sym_quantize(hidden_states, self.fp8_qmax)
        fp8_restored = self._sym_dequantize(fp8_q, fp8_scale)

        if kv_cache is None:
            return fp8_restored

        self.quantize_kv_cache(kv_cache)
        kv_restored = self.dequantize_kv_cache()
        return fp8_restored, kv_restored

    def mse(self, original: torch.Tensor, restored: torch.Tensor) -> torch.Tensor:
        return torch.mean((original.float() - restored.float()) ** 2)


In [ ]:
# 机制测试：分别检查量化闭环、状态、分组、恢复和误差契约。
def test_symmetric_quantization_contract():
    """验证对称量化、反量化和有限值契约。"""
    sim = FP8KVCacheSim(fp8_qmax=127, kv_group_size=4)
    x = torch.tensor([-2.0, 0.0, 1.0, 2.0])
    q, scale = sim._sym_quantize(x, sim.fp8_qmax)
    restored = sim._sym_dequantize(q, scale)
    assert q.dtype == torch.int8
    assert torch.isfinite(scale).all()
    assert torch.isfinite(restored).all()
    assert float((restored - x).abs().max()) < 0.03


def test_fp8_state_contract():
    """验证运行时量化状态保存原始形状。"""
    sim = FP8KVCacheSim(fp8_qmax=127, kv_group_size=4)
    hidden = torch.randn(2, 8)
    sim.quantize_fp8(hidden)
    assert sim.fp8_q.dtype == torch.int8
    assert sim.fp8_shape == tuple(hidden.shape)
    assert sim.dequantize_fp8().shape == hidden.shape


def test_kv_group_partition_contract():
    """验证 KV Cache 分组数量和 scale 形状。"""
    sim = FP8KVCacheSim(fp8_qmax=127, kv_group_size=4)
    kv = torch.randn(2, 3, 7)
    sim.quantize_kv_cache(kv)
    assert sim.kv_q.dtype == torch.int8
    assert sim.kv_scale.shape == (2, 3, 2)
    assert sim.kv_shape == tuple(kv.shape)


def test_kv_dequantization_contract():
    """验证分组 KV Cache 可以恢复原始形状并保持有限值。"""
    sim = FP8KVCacheSim(fp8_qmax=127, kv_group_size=4)
    kv = torch.randn(2, 3, 8)
    sim.quantize_kv_cache(kv)
    restored = sim.dequantize_kv_cache()
    assert restored.shape == kv.shape
    assert torch.isfinite(restored).all()
    assert float((restored - kv).abs().mean()) < 0.03


def test_zero_input_and_error_metric_contract():
    """验证全零输入不会产生非法 scale，且误差指标可计算。"""
    sim = FP8KVCacheSim(fp8_qmax=127, kv_group_size=4)
    hidden = torch.zeros(2, 7)
    kv = torch.zeros(1, 2, 7)
    sim.fit(hidden, kv)
    assert torch.isfinite(sim.dequantize_fp8()).all()
    assert torch.isfinite(sim.dequantize_kv_cache()).all()
    assert sim.kv_scale.shape[-1] == 2
    assert float(sim.mse(hidden, sim.dequantize_fp8())) == 0.0


def run_fp8_kv_cache_tests():
    """汇总五组 FP8/KV Cache 机制测试。"""
    for test in (
        test_symmetric_quantization_contract,
        test_fp8_state_contract,
        test_kv_group_partition_contract,
        test_kv_dequantization_contract,
        test_zero_input_and_error_metric_contract,
    ):
        test()
    print('✅ FP8/KV Cache 机制测试通过：量化、状态、分组、恢复与误差均已验证。')


run_fp8_kv_cache_tests()


## 参考代码与解析

### 代码


In [ ]:
class FP8KVCacheSim(nn.Module):
    """保存量化值、scale 和 shape 的最小运行时量化模拟器。

    `fp8_qmax` 是量化整数范围上限，`kv_group_size` 是 KV Cache 最后一维共享
    一套 scale 的元素数，`eps` 用于避免全零或极小动态范围下的除零。
    """

    def __init__(self, fp8_qmax: int = 127, kv_group_size: int = 64, eps: float = 1e-8):
        super().__init__()
        if kv_group_size <= 0:
            raise ValueError("kv_group_size must be positive")
        self.fp8_qmax = fp8_qmax
        self.kv_group_size = kv_group_size
        self.eps = eps

        self.register_buffer("fp8_q", torch.empty(0, dtype=torch.int8), persistent=False)
        self.register_buffer("fp8_scale", torch.tensor(1.0), persistent=False)
        self.register_buffer("kv_q", torch.empty(0, dtype=torch.int8), persistent=False)
        self.register_buffer("kv_scale", torch.empty(0), persistent=False)
        self.fp8_shape = None
        self.kv_shape = None

    def _sym_quantize(self, x: torch.Tensor, qmax: int):
        x = x.detach().float()
        # TODO 1：由动态范围生成安全 scale，并映射为有界 INT8 值。
        # absmax = ???
        absmax = torch.max(torch.abs(x))
        # scale = ???
        scale = qmax / absmax.clamp_min(self.eps)
        # q = ???
        q = torch.clamp(torch.round(x * scale), -qmax, qmax).to(torch.int8)
        return q, scale

    def _sym_dequantize(self, q: torch.Tensor, scale: torch.Tensor):
        return q.to(scale.dtype) / scale.clamp_min(self.eps)

    def quantize_fp8(self, x: torch.Tensor):
        q, scale = self._sym_quantize(x, self.fp8_qmax)
        self.fp8_q = q
        self.fp8_scale = scale
        self.fp8_shape = tuple(x.shape)
        return q, scale

    def dequantize_fp8(self):
        if self.fp8_shape is None:
            raise RuntimeError("Call quantize_fp8() before dequantize_fp8().")
        return self._sym_dequantize(self.fp8_q, self.fp8_scale)

    def quantize_kv_cache(self, kv_cache: torch.Tensor):
        kv = kv_cache.detach().float()
        if kv.ndim < 2:
            raise ValueError("KV cache should have at least 2 dimensions.")

        last_dim = kv.size(-1)
        # TODO 2：按最后一维向上取整，保留最后不足一组的元素。
        n_groups = (last_dim + self.kv_group_size - 1) // self.kv_group_size
        qkv = torch.zeros_like(kv, dtype=torch.int8)
        scales = torch.zeros(kv.shape[:-1] + (n_groups,), dtype=kv.dtype, device=kv.device)

        flat = kv.reshape(-1, last_dim)
        flat_q = qkv.reshape(-1, last_dim)
        flat_scale = scales.reshape(-1, n_groups)

        for row in range(flat.size(0)):
            for g in range(n_groups):
                start = g * self.kv_group_size
                end = min(start + self.kv_group_size, last_dim)
                chunk = flat[row, start:end]
                if chunk.numel() == 0:
                    continue
                q, scale = self._sym_quantize(chunk, self.fp8_qmax)
                flat_q[row, start:end] = q
                flat_scale[row, g] = scale

        self.kv_q = qkv
        self.kv_scale = scales
        self.kv_shape = tuple(kv.shape)
        return qkv, scales

    def dequantize_kv_cache(self):
        if self.kv_shape is None:
            raise RuntimeError("Call quantize_kv_cache() before dequantize_kv_cache().")

        kv = self.kv_q.to(self.kv_scale.dtype)
        last_dim = kv.size(-1)
        n_groups = self.kv_scale.size(-1)
        flat = kv.reshape(-1, last_dim)
        flat_out = torch.zeros_like(flat, dtype=self.kv_scale.dtype)
        flat_scale = self.kv_scale.reshape(-1, n_groups)

        for row in range(flat.size(0)):
            for g in range(n_groups):
                start = g * self.kv_group_size
                end = min(start + self.kv_group_size, last_dim)
                scale = flat_scale[row, g]
                # TODO 3：使用当前 group 的 scale 恢复对应 KV 区间。
                restored_chunk = self._sym_dequantize(flat[row, start:end], scale)
                flat_out[row, start:end] = restored_chunk

        return flat_out.reshape(self.kv_shape)

    def fit(self, hidden_states: torch.Tensor, kv_cache: torch.Tensor | None = None):
        self.quantize_fp8(hidden_states)
        if kv_cache is not None:
            self.quantize_kv_cache(kv_cache)
        return self

    def forward(self, hidden_states: torch.Tensor, kv_cache: torch.Tensor | None = None):
        fp8_q, fp8_scale = self._sym_quantize(hidden_states, self.fp8_qmax)
        fp8_restored = self._sym_dequantize(fp8_q, fp8_scale)

        if kv_cache is None:
            return fp8_restored

        self.quantize_kv_cache(kv_cache)
        kv_restored = self.dequantize_kv_cache()
        return fp8_restored, kv_restored

    def mse(self, original: torch.Tensor, restored: torch.Tensor) -> torch.Tensor:
        return torch.mean((original.float() - restored.float()) ** 2)


### 解析

**TODO 1：建立对称量化状态**

- `absmax` 描述当前张量的动态范围；`clamp_min(eps)` 让全零或极小张量仍得到有限 scale。
- 量化值经过 `round` 与范围限制后保存为 INT8，恢复时必须使用同一个 scale。

**TODO 2：沿最后一维分组 KV Cache**

- 最后一维通常承载每个 head 的特征维度。向上取整的分组数保证尾部不足 `kv_group_size` 的元素不会丢失。
- 每个前缀位置、每个 group 都保存独立 scale；测试检查了长度为 7 时仍会得到两个 group。

**TODO 3：按组恢复**

- 恢复某段 KV 值时，只能读取同一 row、同一 group 的 scale；混用 scale 会破坏量化状态。
- `fp8_shape`、`kv_shape` 与 MSE 由骨架记录，帮助测试确认恢复形状和误差可比较。

本题使用 INT8 容器展示 scale 与分组机制。真实 FP8 格式、硬件 kernel 与 serving 结果需要由 Step 5 的实测证据判断。


### Step 5：可选 GPU 实验——测量 FP8 / KV Cache 模拟路径

![FP8 KV Cache GPU 机制实验流程](../docs/public/02_PyTorch_Algorithms/41_fp8_kv_gpu_mechanism_flow.svg)

实验从真实模型的 past_key_values 取得一段实际 KV 状态，再比较分组量化的原始字节数、量化字节数、恢复误差、耗时和峰值显存。当前使用 INT8 容器模拟量化闭环，并把 K/V 拼成统一教学张量；它不代表真实 FP8 Tensor Core、backend 内部 KV 布局或 serving 收益，证据等级记为 gpu_simulation_on_real_kv_state。

#### 5.1 环境、输入与固定条件

先确认 CUDA、模型版本、dtype、batch size 和输入长度。结果必须同时记录配置的最大长度和实际 prompt_tokens；比较 group size 或输入长度时，每次只改变一个主要变量。

#### 5.2 执行 KV Cache 量化并保存 JSON

先运行 dry_run 检查环境，再切换到 real_gpu。执行单元只读取 5.1 的固定配置，保存原始/量化字节数、恢复误差、耗时、runtime、失败状态和峰值显存。


In [ ]:
# 5.1 只定义固定 workload；默认不下载模型、不启动 GPU 测量。
from pathlib import Path

RUN_MODE = 'dry_run'  # dry_run / real_gpu
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
PROMPT = 'Explain how KV Cache grows during generation.'
SEED = 42
BATCH_SIZE = 1
SEQ_LEN = 512
KV_GROUP_SIZE = 32
WARMUP = 5
ITERS = 20
OUTPUT_PATH = Path('benchmarks/results/41_fp8_kv_gpu.json')


In [ ]:
import json
import platform
import time
# 5.2 读取 5.1 的固定配置并执行；请先运行上一个配置单元。
NUM_HEADS = 16  # CPU dry_run 的合成 KV 形状参数；real_gpu 会由真实模型状态覆盖。
HEAD_DIM = 64  # CPU dry_run 的合成 KV 形状参数；real_gpu 会由真实模型状态覆盖。

torch.manual_seed(SEED)
cuda_available = torch.cuda.is_available()
if RUN_MODE == 'real_gpu' and not cuda_available:
    raise RuntimeError('RUN_MODE=real_gpu 但 CUDA 不可用，请先完成 GPU 环境预检。')
device = torch.device('cuda' if RUN_MODE == 'real_gpu' else 'cpu')
runtime = {'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
           'cuda_available': cuda_available, 'device': torch.cuda.get_device_name(0) if cuda_available else 'cpu'}

def _sync():
    """确保 CUDA 异步操作完成后再读取计时或显存。"""
    if device.type == 'cuda': torch.cuda.synchronize()

def _measure(fn):
    """测量量化和恢复过程的平均耗时。"""
    for _ in range(WARMUP): fn()
    _sync(); start = time.perf_counter()
    for _ in range(ITERS): fn()
    _sync()
    return round((time.perf_counter() - start) * 1000 / ITERS, 4)

shape = (BATCH_SIZE, NUM_HEADS, SEQ_LEN, HEAD_DIM)
evidence_level = 'environment_preflight' if RUN_MODE == 'dry_run' else 'gpu_simulation_on_real_kv_state'
result = {'stage': evidence_level, 'run_mode': RUN_MODE, 'runtime': runtime, 'json_path': str(OUTPUT_PATH),
          'workload': {'model_id': MODEL_ID, 'batch_size': BATCH_SIZE, 'seq_len': SEQ_LEN,
                       'kv_group_size': KV_GROUP_SIZE, 'state_scope': 'one model KV state'},
          'config': {
    'shape': list(shape), 'kv_group_size': KV_GROUP_SIZE, 'warmup': WARMUP, 'iters': ITERS, 'seed': SEED, 'model_id': MODEL_ID,
}, 'evidence_level': evidence_level, 'baseline': 'FP16 KV state in memory',
   'candidate': 'INT8 container simulation in memory', 'artifact_path': None, 'failure': None}
if RUN_MODE == 'dry_run':
    result['decision'] = {'decision': 'ready_to_measure', 'reason': '仅完成环境与配置检查，尚未运行 GPU KV Cache 测量。'}
else:
    # real_gpu 从真实模型的 past_key_values 读取 KV；cpu 模式保留小型确定性张量。
    if RUN_MODE == 'real_gpu':
        from transformers import AutoModelForCausalLM, AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to(device).eval()
        inputs = tokenizer(PROMPT, return_tensors='pt', truncation=True, max_length=SEQ_LEN).to(device)
        actual_prompt_tokens = int(inputs['input_ids'].shape[1])
        with torch.no_grad(): outputs = model(**inputs, use_cache=True, return_dict=True)
        past = outputs.past_key_values
        if hasattr(past, 'to_legacy_cache'): past = past.to_legacy_cache()
        key, value = past[0][0], past[0][1]
        kv = torch.cat([key, value], dim=1).float()
        shape = tuple(kv.shape)
        del model, outputs, past, inputs
        if device.type == 'cuda': torch.cuda.empty_cache()
    else:
        kv = torch.randn(*shape, device=device)
    result['config'].update({'shape': list(shape), 'prompt_tokens': actual_prompt_tokens if RUN_MODE == 'real_gpu' else None,
                             'state_source': 'real_model_past_key_values' if RUN_MODE == 'real_gpu' else 'synthetic_cpu',
                             'layout_note': 'K/V concatenated on head axis for teaching'})
    sim = FP8KVCacheSim(kv_group_size=KV_GROUP_SIZE).to(device)
    def quantize_and_restore():
        sim.quantize_kv_cache(kv)
        return sim.dequantize_kv_cache()
    latency = _measure(quantize_and_restore)
    restored = sim.dequantize_kv_cache()
    raw_bytes = int(kv.numel() * kv.element_size())
    quant_bytes = int(sim.kv_q.numel() * sim.kv_q.element_size() + sim.kv_scale.numel() * sim.kv_scale.element_size())
    peak = torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None
    result.update({'metrics': {'raw_bytes': raw_bytes, 'quantized_bytes_with_scale': quant_bytes,
        'compression_ratio': round(raw_bytes / quant_bytes, 4), 'mse': round(float(sim.mse(kv, restored)), 8),
        'quantize_restore_latency_ms': latency, 'peak_memory_mb': None if peak is None else round(peak, 2),
        'restored_shape': list(restored.shape)},
        'decision': {'decision': 'measure', 'reason': '仅观察 KV Cache 分组量化的容量、误差和恢复代价；不代表真实 serving 收益。'}})
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(result, ensure_ascii=False, indent=2))

#### 5.3 读取主实验结果

先读取主实验 JSON，核对实际 KV shape、分组大小、量化字节数和恢复误差；实际 prompt 长度与配置最大长度都应保留在记录中。

**FP8 weight-only 探针（可选）**：PyTorch `torchao` 的 `Float8WeightOnlyConfig` 只用于观察 FP8 线性层 API 是否能在当前设备执行；它不测量 KV Cache，也不改变主实验的 KV Cache 证据。探针结果单独保存。


In [ ]:
# 5.3 只读取 5.2 保存的主实验结果；不重新生成 KV Cache。
if OUTPUT_PATH.exists():
    saved = json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
    print({key: saved.get(key) for key in ('workload', 'config', 'metrics', 'failure', 'evidence_level', 'decision')})
else:
    print(f'等待 GPU KV Cache 结果：{OUTPUT_PATH}')


In [ ]:
RUN_FP8_TORCHAO_PROBE = False  # 默认关闭；需要兼容 CUDA / torchao 环境时改为 True
FP8_OUTPUT_PATH = Path('benchmarks/results/41_fp8_torchao_probe.json')

if not RUN_FP8_TORCHAO_PROBE:
    print('torchao FP8 probe skipped; set RUN_FP8_TORCHAO_PROBE=True on a compatible environment.')
else:
    if not torch.cuda.is_available():
        raise RuntimeError('RUN_FP8_TORCHAO_PROBE=True requires CUDA.')
    try:
        from torchao.quantization import Float8WeightOnlyConfig, quantize_
    except ImportError as exc:
        raise ImportError('请安装与当前 PyTorch 兼容的 torchao，再运行 FP8 探针。') from exc
    fp8_probe = nn.Linear(HEAD_DIM, HEAD_DIM, device='cuda', dtype=torch.bfloat16).eval()
    fp8_input = torch.randn(BATCH_SIZE, SEQ_LEN, HEAD_DIM, device='cuda', dtype=torch.bfloat16)
    quantize_(fp8_probe, Float8WeightOnlyConfig())
    torch.cuda.synchronize()
    with torch.inference_mode():
        fp8_output = fp8_probe(fp8_input)
    torch.cuda.synchronize()
    fp8_result = {'json_path': str(FP8_OUTPUT_PATH),
                  'workload': {'batch_size': BATCH_SIZE, 'seq_len': SEQ_LEN, 'head_dim': HEAD_DIM},
                  'baseline': 'BF16 Linear in memory', 'candidate': 'torchao Float8WeightOnlyConfig',
                  'library': 'torchao', 'config': 'Float8WeightOnlyConfig',
                  'input_shape': list(fp8_input.shape), 'output_shape': list(fp8_output.shape),
                  'device': torch.cuda.get_device_name(0),
                  'evidence_level': 'mature_library_weight_only_api_probe',
                  'decision': 'weight_only_api_path_executed', 'artifact_path': None, 'failure': None}
    FP8_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    FP8_OUTPUT_PATH.write_text(json.dumps(fp8_result, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(fp8_result, ensure_ascii=False, indent=2))


#### 5.4 GPU 实验结果记录

量化字节数包含 scale 元数据；真实 FP8 kernel、KV Cache serving 和端到端质量需要转到对应 backend 项目验证。

| role | baseline / candidate | artifact | shape | kv_group_size | runtime | 原始字节 | 量化字节（含 scale） | 压缩比 | MSE | peak memory (MB) | failure | evidence level | decision |
|---|---|---|---|---:|---|---:|---:|---:|---:|---:|---|---|---|
| reference | baseline | FP16 KV state / JSON path |  |  |  |  |  |  |  |  |  | gpu_simulation_on_real_kv_state |  |
| simulated | candidate | INT8 container in memory / JSON path |  |  |  |  |  |  |  |  |  | gpu_simulation_on_real_kv_state | accept / tune / reject |
| FP8 weight-only probe | candidate probe | torchao Linear module / JSON path |  |  |  |  |  |  |  |  |  | mature_library_weight_only_api_probe | API path only；不作为 KV 证据 |
| serving | candidate backend | KV Cache backend artifact / result JSON |  |  |  |  |  |  |  |  |  | backend_benchmark_pending | accept / tune / reject |

## 相关阅读

完成 FP8 scale、KV Cache 分组和误差检查后，可以继续阅读 FP8 格式、缓存调度和真实部署 backend。

- [FP8 原论文：FP8 Formats for Deep Learning](https://arxiv.org/abs/2209.05433)
- [torchao 官方工作流：Inference Quantization](https://docs.pytorch.org/ao/stable/workflows/inference.html)
- [NVIDIA Transformer Engine 官方仓库](https://github.com/NVIDIA/TransformerEngine)
- [37. KV Cache Scheduling | KV Cache 调度](./37_KV_Cache_Scheduling.ipynb)
- [67. Quantized Inference and Deployment | 量化推理与部署](./67_Quantized_Inference_and_Deployment.ipynb)
- [75. Memory Budget Compression Project | 显存预算压缩项目](./75_Memory_Budget_Compression_Project.ipynb)
